# Remedial Assessment - Solutions

## Question-1: Running SQL Commands in Python

### PostgreSQL Environment Variables
Please setup postreSQL environment variables as below: 

- **PGHOST** → The host where PostgreSQL is running.  
  For local setups this is usually `localhost`.  
- **PGPORT** → The port PostgreSQL listens on.  
  Default is `5432`.  
- **PGDATABASE** → The database name you created for this lab, e.g. `ntl_pop`.  
- **PGUSER** → Your PostgreSQL username.  
  This is usually the same as your system username when running locally.  

⚠️ **Note:** On a local setup, you usually don’t need a password if PostgreSQL is configured to trust local connections.  
In that case, we will not use `PGPASSWORD`.  

In [27]:
import os
import requests
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

In [2]:
# 1) Load environment variables
# The .env file should be in the repo root with keys:
# PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD

# looks for .env in current directory by default
load_dotenv()  

PGHOST = os.getenv("PGHOST")
PGPORT = os.getenv("PGPORT", "5432")
PGDATABASE = os.getenv("PGDATABASE")
PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")

In [3]:
print("PostgreSQL Connection Settings:")
print(f"Host: {PGHOST}")
print(f"Port: {PGPORT}")
print(f"Database: {PGDATABASE}")
print(f"User: {PGUSER}")
print(f"Password: {'[SET]' if PGPASSWORD else '[NOT SET]'}")

PostgreSQL Connection Settings:
Host: localhost
Port: 5432
Database: ntl_pop
User: dmatekenya
Password: [NOT SET]


### Question-1 Instructions
We are using the **ntl_pop** database as created in this [lab work](https://docs.google.com/document/d/1yPYsS6XlCnwYQopkMmiE4EBCQU9J3sDOh1w7DWaxwr0/edit?usp=sharing).
1. Create connection string to your database 
2. Create sql engine 
3. Use peek function below to retrieve results from your database
4. Print the Top 5 brightest cells in 2022 as shown in cell below:
   
   ``` print("\n--- Top 5 brightest cells in 2022 ---")
brightest = peek("""
SELECT cell_id, ntl_mean
FROM ntl_annual
WHERE year = 2022
ORDER BY ntl_mean DESC
LIMIT 5;
""")
print(brightest)
```

## Question-1-1 [2 pts]

In [4]:
# Create SQLAlchemy engine (no password needed for local connections)
connection_string = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
print("Connection psql string:", connection_string)

Connection psql string: postgresql+psycopg2://dmatekenya:@localhost:5432/ntl_pop


## Question-1-2 [2 pts]

In [5]:
engine = create_engine(
    connection_string,
    pool_pre_ping=True,
)

# Test the connection
with engine.connect() as conn:
    who = conn.execute(text("SELECT current_user, current_database();")).fetchone()
    print("Connected as:", who)

Connected as: ('dmatekenya', 'ntl_pop')


In [16]:
def peek(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a pandas DataFrame."""
    return pd.read_sql(sql, engine)

In [18]:
def peek_v2(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a pandas DataFrame."""
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        return pd.DataFrame(result.fetchall(), columns=result.keys())


## Question-1-3 [6 pts]

Use the `peek` function defined above to retrieve results from the database. Demonstrate that you can successfully query any table in the **ntl_pop** database. You may query any table and retrieve any type of results. 

**Note:** If the original `peek` function encounters any issues, you must debug and update it until it works correctly.

Below are examples of acceptable queries and outputs:


In [21]:
# Example 1: Get a sample of records from ntl_annual table
print("Example 1: Sample records from ntl_annual")
sample_data = peek_v2("""
SELECT * FROM ntl_annual
LIMIT 10
""")
print(sample_data)

Example 1: Sample records from ntl_annual
   id        cell_id  ntl_min  ntl_max  ntl_mean  ntl_median  ntl_sum  \
0   1  RWA.1.1.1.1_1      0.0      0.0       0.0        26.0      0.0   
1   2  RWA.1.1.1.2_1      0.0      0.0       0.0        27.0      0.0   
2   3  RWA.1.1.1.3_1      0.0      0.0       0.0        27.0      0.0   
3   4  RWA.1.1.1.4_1      0.0      0.0       0.0        43.0      0.0   
4   5  RWA.1.1.2.1_1      0.0      0.0       0.0        64.0      0.0   
5   6  RWA.1.1.2.2_1      0.0      0.0       0.0        42.0      0.0   
6   7  RWA.1.1.2.3_1      0.0      0.0       0.0        38.0      0.0   
7   8  RWA.1.1.2.4_1      0.0      0.0       0.0        76.0      0.0   
8   9  RWA.1.1.2.5_1      0.0      0.0       0.0        51.0      0.0   
9  10  RWA.1.1.3.1_1      0.0      0.0       0.0        28.0      0.0   

   pixel_count                                raster_filename  year  
0          0.0  VNP46A4.A2012001.h20v09.001.2021124123213.tif  2012  
1          0.0

In [22]:
# Example 2: Get summary statistics by year
print("\nExample 2: Average nighttime light intensity by year")
yearly_avg = peek_v2("""
SELECT year, AVG(ntl_mean) as avg_ntl, COUNT(*) as cell_count
FROM ntl_annual
GROUP BY year
ORDER BY year;
""")
print(yearly_avg)


Example 2: Average nighttime light intensity by year
    year   avg_ntl  cell_count
0   2012  0.207097        4338
1   2013  0.253869        4338
2   2014  0.257401        4338
3   2015  0.244143        4338
4   2016  0.257303        4338
5   2017  0.262365        4338
6   2018  0.284452        4338
7   2019  0.299609        4338
8   2020  0.305707        4338
9   2021  0.358215        4338
10  2022  0.453329        4338
11  2023  0.564303        4338


In [23]:
# Example 3: Top 5 brightest cells in 2022
print("\nExample 3: Top 5 brightest cells in 2022")
brightest = peek_v2("""
SELECT cell_id, ntl_mean
FROM ntl_annual
WHERE year = 2022
ORDER BY ntl_mean DESC
LIMIT 5;
""")
print(brightest)


Example 3: Top 5 brightest cells in 2022
          cell_id ntl_mean
0   RWA.5.3.1.3_1     None
1   RWA.5.3.6.6_1     None
2   RWA.5.3.1.2_1     None
3  RWA.2.2.15.1_1     None
4   RWA.5.3.7.2_1     None


## Question-1-4 [10 pts]

For this question, the trainee needs to return the same exact results as shown in **CELL INDEX 18** below in order to get 10 points. You can assign points based on the following:

- **Correct structure of SQL** - 2 pts (Basically they copied it correctly)
- **Exclude NULL** - 4 pts (They need to add `WHERE ntl_mean IS NOT NULL` or similar condition to the query)
- **Numbers match the expected results** - 4 pts (The output should match the dataframe with cell_ids and ntl_mean values shown below)

In [25]:
print("\n--- Top 5 brightest cells in 2022 ---")
brightest = peek_v2("""
SELECT cell_id, ntl_mean
FROM ntl_annual
WHERE year = 2022 AND ntl_mean IS NOT NULL
ORDER BY ntl_mean DESC
LIMIT 5;
""")
print(brightest)


--- Top 5 brightest cells in 2022 ---
         cell_id   ntl_mean
0  RWA.5.3.6.5_1  50.800000
1  RWA.5.3.9.2_1  48.400000
2  RWA.5.3.9.3_1  45.431580
3  RWA.5.3.9.4_1  43.300000
4  RWA.5.3.6.7_1  43.166668


### <span style="color: #ab2e98ff; font-weight: bold;">Total Points for Question-1: 20</span>

-----

## Question-2: Open Weather API
Given a city name, write a function that gets current weather data and forecasts and then save that information as a CSV file. You can use code below as staretr code. We are only interested in the following variables:
- **Weather:** temp, min and max temperature, humidity
- **Other:** Date, with no time

In order to demonstrate that the code works, retrieve data for the following cities: Lilongwe, Blantyre, Kigali and Zomba. Save the retrieve data into a CSV file.

```
BASE_URL = "https://api.openweathermap.org/data/2.5/"

API_KEY = os.getenv("OPENWEATHER_API_KEY")

# Create full URL for current weather in Lilongwe
category = 'weather'
city_name = "Lilongwe"
full_url = "{}{}?q={}&APPID={}".format(BASE_URL, category, city_name, API_KEY)
print(full_url)
```



### <span style="color: #ff6b35; font-weight: bold;">Total Points for Question-2: 30</span>


1. **Function Implementation [15 pts]**
    - Implement a `get_weather_data` function to retrieve weather data from the OpenWeather API
    - Function should accept city name as parameter
    - Function should return a dictionary or structured data with required fields
    - Use of `units=metric` parameter for temperature in Celsius
    - Extract only the required fields: temp, temp_min, temp_max, humidity, and date (YYYY-MM-DD format, no time)
    - A sample implementation is provided in the cells below

2. **API Key Usage [2 pts]**
    - Correctly retrieve API key from environment variables using `os.getenv()`. They can also hard code the key.

3. **Cities List [2 pts]**
    - Define a list containing the four required cities: ['Lilongwe', 'Blantyre', 'Kigali', 'Zomba']
    - Iterate through the list to fetch data for all cities

4. **Results Display and CSV Export [10 pts]**
    - Successfully retrieve data for all 4 cities (8 pts)
    - DataFrame contains correct column names: city, date, temp, temp_min, temp_max, humidity (2 pts)
    - Save results to CSV file using `to_csv()` method (bonus if implemented)

5. **Data Quality [1 pt bonus]**
    - Date format is clean (YYYY-MM-DD) without timestamp
    - Temperature values are in Celsius (metric units)
    - All expected fields are present and correctly labeled

----

In [46]:
def get_weather_data(city_name, base_url, api_key):
    """
    Get current weather data for a given city and return relevant information.
    
    Parameters:
    -----------
    city_name : str
        Name of the city to get weather data for
        
    Returns:
    --------
    dict : Dictionary containing weather information
    """
    category = 'weather'
    full_url = "{}{}?q={}&APPID={}&units=metric".format(base_url, category, city_name, api_key)
    print("Requesting URL:", full_url)
    
    try:
        response = requests.get(full_url)
        response.raise_for_status()
        data = response.json()
        
        # Extract relevant information
        weather_info = {
            'city': city_name,
            'date': datetime.fromtimestamp(data['dt']).strftime('%Y-%m-%d'),
            'temp': data['main']['temp'],
            'temp_min': data['main']['temp_min'],
            'temp_max': data['main']['temp_max'],
            'humidity': data['main']['humidity']
        }
        
        return weather_info
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {city_name}: {e}")
        return None



In [47]:
BASE_URL = "https://api.openweathermap.org/data/2.5/"
API_KEY = os.getenv("OPENWEATHER_API_KEY")


# List of cities to retrieve weather data for
cities = ['Lilongwe', 'Blantyre', 'Kigali', 'Zomba']

# Collect weather data for all cities
weather_data_list = []
for city in cities:
    print(f"Fetching weather data for {city}...")
    weather_info = get_weather_data(city, BASE_URL, API_KEY)
    if weather_info:
        weather_data_list.append(weather_info)

# Convert to DataFrame and save to CSV
weather_df = pd.DataFrame(weather_data_list)
print("\nWeather Data:")
print(weather_df)

# Save to CSV
csv_filename = 'weather_data.csv'
weather_df.to_csv(csv_filename, index=False)
print(f"\nWeather data saved to {csv_filename}")

Fetching weather data for Lilongwe...
Requesting URL: https://api.openweathermap.org/data/2.5/weather?q=Lilongwe&APPID=15b6dcf6a90d01c149cd46da48646612&units=metric
Fetching weather data for Blantyre...
Requesting URL: https://api.openweathermap.org/data/2.5/weather?q=Blantyre&APPID=15b6dcf6a90d01c149cd46da48646612&units=metric
Fetching weather data for Kigali...
Requesting URL: https://api.openweathermap.org/data/2.5/weather?q=Kigali&APPID=15b6dcf6a90d01c149cd46da48646612&units=metric
Fetching weather data for Zomba...
Requesting URL: https://api.openweathermap.org/data/2.5/weather?q=Zomba&APPID=15b6dcf6a90d01c149cd46da48646612&units=metric

Weather Data:
       city        date   temp  temp_min  temp_max  humidity
0  Lilongwe  2025-12-20  20.66     20.66     20.66        86
1  Blantyre  2025-12-20  20.25     20.25     20.25        88
2    Kigali  2025-12-20  19.29     19.29     19.71        72
3     Zomba  2025-12-20  19.73     19.73     19.73        94

Weather data saved to weather

### <span style="color: #ff6b35; font-weight: bold;">Total Points-50</span>
---